In [1]:
import pandas as pd
df=pd.read_csv("../data/processed/final_clean_data.csv")

In [2]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [3]:
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['dayofweek'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)


In [4]:
df[['date','day','month','dayofweek','is_weekend']].head()


,date,day,month,dayofweek,is_weekend
0,2024-01-01,1,1,0,0
1,2024-01-01,1,1,0,0
2,2024-01-01,1,1,0,0
3,2024-01-01,1,1,0,0
4,2024-01-01,1,1,0,0


In [5]:
df = df.sort_values(['district', 'commodity', 'date']).reset_index(drop=True)


In [6]:
lags = [1, 7, 14]

for lag in lags:
    df[f'modal_price_lag_{lag}'] = (
        df.groupby(['district','commodity'])['modal_price']
          .shift(lag)
    )


In [7]:
df[
    (df['district']=='Ahmednagar') &
    (df['commodity']=='Tomato')
][[
    'date','modal_price','modal_price_lag_1','modal_price_lag_7'
]].head(20)


,date,modal_price,modal_price_lag_1,modal_price_lag_7
29884,2024-01-01,1850.0,NaN,NaN
29885,2024-01-02,2100.0,1850.0,NaN
29886,2024-01-03,2000.0,2100.0,NaN
29887,2024-01-03,2000.0,2000.0,NaN
29888,2024-01-04,2000.0,2000.0,NaN
29889,2024-01-04,2000.0,2000.0,NaN
29890,2024-01-05,1700.0,2000.0,NaN
29891,2024-01-05,2000.0,1700.0,1850.0
29892,2024-01-06,1700.0,2000.0,2100.0
29893,2024-01-06,1600.0,1700.0,2000.0


In [8]:
df.to_csv("../data/processed/features_step1_lags.csv", index=False)

In [9]:
df = (
    df.groupby(['date','district','commodity'], as_index=False)
      .agg({
          'modal_price': 'mean',
          'min_price': 'min',
          'max_price': 'max',
          'temp_max': 'mean',
          'temp_min': 'mean',
          'rainfall': 'mean',
          'wind_speed': 'mean'
      })
)


In [10]:
df = df.sort_values(['district','commodity','date']).reset_index(drop=True)

In [11]:
lags = [1, 7, 14]

for lag in lags:
    df[f'modal_price_lag_{lag}'] = (
        df.groupby(['district','commodity'])['modal_price']
          .shift(lag)
    )


In [12]:
df[
    (df['district']=='Ahmednagar') &
    (df['commodity']=='Tomato')
][[
    'date','modal_price','modal_price_lag_1','modal_price_lag_7'
]].head(20)


,date,modal_price,modal_price_lag_1,modal_price_lag_7
12573,2024-01-01,1850.000000,NaN,NaN
12574,2024-01-02,2100.000000,1850.000000,NaN
12575,2024-01-03,2000.000000,2100.000000,NaN
12576,2024-01-04,2000.000000,2000.000000,NaN
12577,2024-01-05,1850.000000,2000.000000,NaN
12578,2024-01-06,1625.000000,1850.000000,NaN
12579,2024-01-07,1733.333333,1625.000000,NaN
12580,2024-01-08,2200.000000,1733.333333,1850.000000
12581,2024-01-09,1150.000000,2200.000000,2100.000000
12582,2024-01-10,1650.000000,1150.000000,2000.000000


In [13]:
df.head()

,date,district,commodity,modal_price,min_price,max_price,temp_max,temp_min,rainfall,wind_speed,modal_price_lag_1,modal_price_lag_7,modal_price_lag_14
0,2024-09-23,Ahmednagar,Amla (Nelli Kai),1250.0,500.0,2000.0,27.3,21.2,6.2,14.1,NaN,NaN,NaN
1,2024-09-25,Ahmednagar,Amla (Nelli Kai),2250.0,1500.0,3000.0,26.1,20.3,21.9,13.5,1250.0,NaN,NaN
2,2024-09-26,Ahmednagar,Amla (Nelli Kai),1000.0,500.0,1500.0,26.9,20.4,6.5,19.2,2250.0,NaN,NaN
3,2024-10-12,Ahmednagar,Amla (Nelli Kai),1750.0,500.0,3000.0,28.2,20.3,6.7,14.2,1000.0,NaN,NaN
4,2024-10-17,Ahmednagar,Amla (Nelli Kai),1500.0,1000.0,2000.0,27.6,20.0,1.9,14.7,1750.0,NaN,NaN


In [14]:
df_lagged = df.dropna().reset_index(drop=True)

In [15]:
df.to_csv("../data/processed/features_step1_lags_clean.csv", index=False)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180013 entries, 0 to 180012
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date                180013 non-null  datetime64[ns]
 1   district            180013 non-null  object        
 2   commodity           180013 non-null  object        
 3   modal_price         180013 non-null  float64       
 4   min_price           180013 non-null  float64       
 5   max_price           180013 non-null  float64       
 6   temp_max            154403 non-null  float64       
 7   temp_min            154403 non-null  float64       
 8   rainfall            154403 non-null  float64       
 9   wind_speed          154403 non-null  float64       
 10  modal_price_lag_1   178683 non-null  float64       
 11  modal_price_lag_7   171611 non-null  float64       
 12  modal_price_lag_14  164171 non-null  float64       
dtypes: datetime64[ns](1), float64

In [17]:
lag_cols = [
    'modal_price_lag_1',
    'modal_price_lag_7',
    'modal_price_lag_14'
]

df_model = df.dropna(subset=lag_cols).reset_index(drop=True)


In [18]:
weather_cols = ['temp_max', 'temp_min', 'rainfall', 'wind_speed']

df_model[weather_cols] = (
    df_model
    .groupby('district')[weather_cols]
    .transform(lambda x: x.interpolate(method='linear'))
)

# Edge fallback (start / end gaps)
df_model[weather_cols] = df_model[weather_cols].bfill()
df_model[weather_cols] = df_model[weather_cols].ffill()


In [19]:
# 1. Lag features
df_model[lag_cols].isna().sum()

# 2. Weather features
df_model[weather_cols].isna().sum()

# 3. Row count sanity
print(df.shape[0], "→", df_model.shape[0])

# 4. Duplicate check
df_model.duplicated(subset=['date', 'district', 'commodity']).sum()


180013 → 164171


np.int64(0)

In [20]:
df_model.head()

,date,district,commodity,modal_price,min_price,max_price,temp_max,temp_min,rainfall,wind_speed,modal_price_lag_1,modal_price_lag_7,modal_price_lag_14
0,2024-10-12,Ahmednagar,Apple,10000.0,7000.0,13000.0,28.2,20.3,6.7,14.2,8500.0,10750.0,11500.0
1,2024-10-13,Ahmednagar,Apple,9500.0,6000.0,13000.0,28.6,20.2,15.0,14.6,10000.0,10375.0,11500.0
2,2024-10-15,Ahmednagar,Apple,8000.0,6000.0,10000.0,28.3,19.7,1.0,11.0,9500.0,9750.0,11250.0
3,2024-10-17,Ahmednagar,Apple,8000.0,6000.0,10000.0,27.6,20.0,1.9,14.7,8000.0,8000.0,11500.0
4,2024-10-19,Ahmednagar,Apple,8000.0,6000.0,10000.0,28.2,19.7,17.1,12.7,8000.0,9500.0,11500.0


In [23]:
df_model.to_csv(
    "../data/processed/phase2_feature_complete.csv",
    index=False
)
